# Barkley pipe flow - Phase 1: continuous regimes (Figs 1-2)

Reproduces Barkley (2011) Fig. 1 profiles and Fig. 2 phase plane with the
continuous two-PDE model solved by the method of lines. Runs on a free Colab
CPU instance in a couple of minutes.


In [ ]:
# --- environment setup (works locally and on Google Colab) ---
try:
    import barkley_pipe  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/yuvrajdabhi2212/barkley-pipe-flow.git"], check=True)
import matplotlib.pyplot as plt
import numpy as np

## Equilibrium puff - Fig. 1(d), $r = 0.7$ (excitable)

In [ ]:
from barkley_pipe.continuous import PeriodicGrid, ContinuousParams, puff_seed, simulate
from barkley_pipe.plotting import plot_profile

grid = PeriodicGrid(n=2000, length=400.0)
puff = simulate(grid, ContinuousParams(r=0.7),
                puff_seed(grid, width=5, q_amplitude=1.0, u_dip=0.6),
                (0.0, 300.0), n_snapshots=151)
ax = plot_profile(puff, recenter=True); ax.set_xlim(150, 260); plt.show()

Localized turbulence, sharp front, long downstream `u`-recovery to 1.

## Expanding slug - Fig. 1(f), $r = 1.0$ (bistable)

In [ ]:
gs = PeriodicGrid(n=2400, length=800.0)
slug = simulate(gs, ContinuousParams(r=1.0),
                puff_seed(gs, center=400, width=5, q_amplitude=1.0, u_dip=0.6),
                (0.0, 250.0), n_snapshots=126)
ax = plot_profile(slug, recenter=True); ax.set_xlim(200, 600); plt.show()

Wide turbulent **plateau** at the bistable turbulent fixed point $(q,u)\approx(1.34,0.13)$ - contrast the plateau-less puff above.

## Phase plane with analytic nullclines - Fig. 2

In [ ]:
from barkley_pipe.plotting import plot_phase_plane, recenter_snapshot
from barkley_pipe.nullclines import fixed_points

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, res, r, lab in [(axes[0], puff, 0.7, "puff"), (axes[1], slug, 1.0, "slug")]:
    qf, uf = recenter_snapshot(res.q[-1], res.u[-1], res.grid.length)
    plot_phase_plane(r, trajectory=(qf, uf), q_max=2.0, ax=ax, label=lab)
    fp = fixed_points(r); ax.scatter(fp[:, 0], fp[:, 1], c="k", s=55, zorder=5)
plt.tight_layout(); plt.show()

At $r=0.7$ the nullclines meet only at laminar $(0,1)$ - one fixed point,
excitable. At $r=1.0$ they meet at three points (laminar, saddle, turbulent
node); the slug trajectory rests on the turbulent node. The critical proxy is
$r_c=\varepsilon_2/(\varepsilon_1+\varepsilon_2)\approx0.833$.
